In [40]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn import metrics
import pandas as pd
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem

In [2]:
from tdc.single_pred import Tox
data = Tox(name = 'LD50_Zhu')
split = data.get_split()

Downloading...
100%|██████████| 707k/707k [00:00<00:00, 9.03MiB/s]
Loading...
Done!


In [11]:
train_data = split["train"]
val_data = split['valid']
test_data= split['test']

In [25]:
train_data.head()

,Drug_ID,Drug,Y
0,"Methane, tribromo-",BrC(Br)Br,2.343
1,Bromoethene (9CI),C=CBr,2.330
2,"1,1'-Biphenyl, hexabromo-",Brc1ccc(-c2ccc(Br)c(Br)c2Br)c(Br)c1Br,1.465
3,"Isothiocyanic acid, p-bromophenyl ester",S=C=Nc1ccc(Br)cc1,2.729
4,"Benzene, bromo-",Brc1ccccc1,1.765


In [13]:
try_smiles = train_data['Drug'][0]

In [15]:
mol_obj = Chem.MolFromSmiles(try_smiles)

In [ ]:
np.array(AllChem.GetMorganFingerprintAsBitVect(mol_obj, radius = 4, nBits=512))

In [34]:
def create_dataset(data):
    fp_lst = []
    smiles_lst = list(data['Drug'])
    y_data = list(data['Y'])
    for smiles in smiles_lst:
        fp_lst.append(np.array(AllChem.GetMorganFingerprintAsBitVect(Chem.MolFromSmiles(smiles), radius = 4, nBits=512)))
    return fp_lst, y_data

In [35]:
train_data_ready, train_y= create_dataset(train_data)
valid_data_ready, valid_y= create_dataset(val_data)
test_data_ready, test_y = create_dataset(test_data)

In [46]:
model = RandomForestRegressor(random_state=42).fit(train_data_ready, train_y)
model2 =SVR().fit(train_data_ready, train_y)

In [47]:
prediction1 = model.predict(test_data_ready)
prediction2 = model2.predict(test_data_ready)


In [48]:
rf_mae = metrics.mean_absolute_error(test_y, prediction1)
svm_mae = metrics.mean_absolute_error(test_y, prediction2)

In [ ]:
print(f"random_forest mae: {rf_mae} \nsvm_mae: {svm_mae}")

random_forest mae: 0.48088916085362765 
svm_mae: 0.47783122121034666


In [44]:
metrics.mean_squared_error(test_y, prediction1)

0.416369215763809